# Reading Model Docs: Context Windows, Pricing, Limits

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/05-reading-model-docs.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You pick a model, write a prompt, and ship to production. Three weeks later you get paged: requests are failing at 2 a.m. with a 429 error. You investigate and discover the model you chose has a 4 requests-per-minute limit on your tier. Or: your document-processing pipeline was working fine on short reports but explodes on the quarterly 120-page PDF because you assumed "200k context window" meant you could send a 200k-token document and get a 200k-token summary back. You cannot.

Both failures came from the same root cause: you picked a model without reading the docs. The model card answers four questions that determine whether a model is viable for your use case before you write a single li...

## The Concept

### The Annotated Model Card

A model listing on any provider's page has five sections that matter for production. Here is what each field means and why it matters.

```
+------------------------------------------------------------------+
| MODEL CARD: claude-3-5-haiku-20241022                            |
+------------------------------------------------------------------+
| CONTEXT WINDOW          200,000 tokens   <-- max INPUT you send  |
|   (input + output combined budget)                               |
|                                                                  |
| MAX OUTPUT TOK...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 05: Reading Model Docs - Context Windows, Pricing, Limits
Phase 00: Setup and Mindset

Reads a hardcoded model spec dict and computes:
  - Max document size you can fit in the context window
  - Cost per 1000 requests at typical token budgets
  - Days until deprecation

No API key required.
"""

from datetime import date, datetime

# ---------------------------------------------------------------------------
# Model spec dicts
# Source: https://www.anthropic.com/pricing  (verify before shipping)
# ---------------------------------------------------------------------------

HAIKU_SPEC = {
    "model_id": "claude-3-5-haiku-20241022",
    "context_window_tokens": 200_000,
    "max_output_tokens": 8_192,
    "pricing": {
        "input_per_million": 0.80,
        "output_per_million": 4.00,
        "cache_read_per_million": 0.08,
        "cache_write_per_million": 1.00,
    },
    "rate_limits": {
        "rpm": 50,
        "tpm": 50_000,
        "rpday": 1_000,
    },
    "deprecation_date": "2025-12-01",
    "modalities": ["text"],
    "knowledge_cutoff": "2024-07",
}

SONNET_SPEC = {
    "model_id": "claude-sonnet-4-5",
    "context_window_tokens": 200_000,
    "max_output_tokens": 64_000,
    "pricing": {
        "input_per_million": 3.00,
        "output_per_million": 15.00,
        "cache_read_per_million": 0.30,
        "cache_write_per_million": 3.75,
    },
    "rate_limits": {
        "rpm": 50,
        "tpm": 80_000,
        "rpday": 5_000,
    },
    "deprecation_date": None,
    "modalities": ["text", "vision"],
    "knowledge_cutoff": "2024-04",
}

### Helpers

In [ ]:
TOKENS_PER_WORD = 1.33  # rough approximation for English prose


def max_document_words(spec: dict, reserved_for_output: int = 1000) -> int:
    """
    How many words fit in a single request?

    We subtract a system prompt budget and the reserved output tokens
    so the returned number is a safe estimate for the document payload.
    """
    system_prompt_tokens = 200
    usable_input_tokens = (
        spec["context_window_tokens"]
        - reserved_for_output
        - system_prompt_tokens
    )
    return int(usable_input_tokens / TOKENS_PER_WORD)


def max_output_words(spec: dict) -> int:
    """Hard cap on response length regardless of context window size."""
    return int(spec["max_output_tokens"] / TOKENS_PER_WORD)


def cost_per_request(
    spec: dict,
    avg_input_tokens: int,
    avg_output_tokens: int,
    cache_hit_rate: float = 0.0,
) -> float:
    """
    Estimated USD cost for one API request.

    Args:
        spec: Model spec dict.
        avg_input_tokens: Tokens in the prompt (system + user).
        avg_output_tokens: Tokens in the response.
        cache_hit_rate: Fraction of input tokens served from cache (0.0 to 1.0).

    Returns:
        Estimated cost in USD.
    """
    pricing = spec["pricing"]
    fresh_input = avg_input_tokens * (1 - cache_hit_rate)
    cached_input = avg_input_tokens * cache_hit_rate

    input_cost = (fresh_input / 1_000_000) * pricing["input_per_million"]
    cache_cost = (cached_input / 1_000_000) * pricing["cache_read_per_million"]
    output_cost = (avg_output_tokens / 1_000_000) * pricing["output_per_million"]

    return input_cost + cache_cost + output_cost


def cost_per_thousand_requests(
    spec: dict,
    avg_input_tokens: int,
    avg_output_tokens: int,
    cache_hit_rate: float = 0.0,
) -> float:
    """Scale a per-request cost to 1,000 requests."""
    return cost_per_request(spec, avg_input_tokens, avg_output_tokens, cache_hit_rate) * 1_000


def days_until_deprecation(spec: dict) -> int | None:
    """
    Days until this model version stops accepting requests.

    Returns None if no deprecation date is set.
    Returns a negative number if the date has already passed.
    """
    if not spec.get("deprecation_date"):
        return None
    dep_date = datetime.strptime(spec["deprecation_date"], "%Y-%m-%d").date()
    today = date.today()
    return (dep_date - today).days

### Report

In [ ]:
def print_model_report(spec: dict) -> None:
    """Print a production-readiness summary for one model spec."""
    print(f"\n{'='*62}")
    print(f"  {spec['model_id']}")
    print(f"{'='*62}")

    max_words = max_document_words(spec)
    max_out = max_output_words(spec)

    print(f"\nCapacity")
    print(f"  Context window : {spec['context_window_tokens']:>10,} tokens  (input budget)")
    print(f"  Max output     : {spec['max_output_tokens']:>10,} tokens  (response cap)")
    print(f"  Max doc size   : ~{max_words:>8,} words   (~{max_words/250:.0f} pages)")
    print(f"  Max output     : ~{max_out:>8,} words")
    print()

    # Cost estimates at three usage tiers
    short_req_cost = cost_per_thousand_requests(spec, 500, 200)
    long_req_cost = cost_per_thousand_requests(spec, 5_000, 500)
    cached_req_cost = cost_per_thousand_requests(spec, 5_000, 500, cache_hit_rate=0.8)

    print(f"Pricing (per 1,000 requests)")
    print(f"  Short prompts  (~500 in / 200 out tokens)   : ${short_req_cost:.2f}")
    print(f"  Long prompts   (~5k in  / 500 out tokens)   : ${long_req_cost:.2f}")
    print(f"  Long + 80% cache hit (same lengths)         : ${cached_req_cost:.2f}")
    savings_pct = (1 - cached_req_cost / long_req_cost) * 100
    print(f"  Cache savings  : {savings_pct:.0f}% reduction")
    print()

    rl = spec["rate_limits"]
    print(f"Rate limits")
    print(f"  RPM   : {rl['rpm']}")
    print(f"  TPM   : {rl['tpm']:,}")
    print(f"  RPDAY : {rl.get('rpday', 'unlimited')}")
    print()

    days = days_until_deprecation(spec)
    if days is None:
        dep_msg = "No deprecation date set"
    elif days < 0:
        dep_msg = f"ALREADY DEPRECATED ({abs(days)} days ago) - migrate immediately"
    elif days < 90:
        dep_msg = f"WARNING: {days} days remaining - plan migration now"
    elif days < 180:
        dep_msg = f"{days} days remaining - schedule migration"
    else:
        dep_msg = f"{days} days remaining"
    print(f"Deprecation: {dep_msg}")

    print(f"\nModalities : {', '.join(spec['modalities'])}")
    print(f"Knowledge  : cutoff {spec.get('knowledge_cutoff', 'unknown')}")


def compare_for_long_output(spec_a: dict, spec_b: dict) -> None:
    """
    Print a focused comparison showing which model supports longer outputs.
    Useful when your use case requires generating long documents.
    """
    print(f"\n{'='*62}")
    print("  Long-output comparison")
    print(f"{'='*62}")
    for spec in [spec_a, spec_b]:
        print(f"\n  {spec['model_id']}")
        print(f"    Max output : {spec['max_output_tokens']:,} tokens "
              f"(~{max_output_words(spec):,} words)")
        cost = cost_per_thousand_requests(spec, 5_000, spec["max_output_tokens"])
        print(f"    Cost at max output (1k req): ${cost:.2f}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Demo

In [ ]:
print_model_report(HAIKU_SPEC)
print_model_report(SONNET_SPEC)
compare_for_long_output(HAIKU_SPEC, SONNET_SPEC)

print("\n\nKey insight:")
print("  A 200k context window with an 8k max output means you can READ")
print("  a 196k-token document but only WRITE an 8k-token response.")
print("  Context window and max output are separate, independent limits.")

---

## Self-Check

Answer these without running code first:

**1. Can this model handle the legal contract feature as described?**

- A. No - the 100,000-token input exceeds the 200k context window.
- B. Yes - the input fits the context window and the 2,000-token output fits the 8,192 max output.
- C. No - context windows and max output are the same limit, so 100k input leaves only 100k for output.
- D. Yes - but only if you enable prompt caching to fit the document.

**2. What is the blocking constraint for this use case?**

- A. The 200k context window - the briefing is too short to fill it properly.
- B. The 8,192 max output tokens - the desired 10,000-token output exceeds this hard cap.
- C. The RPM rate limit - long generation takes too long per request.
- D. The knowledge cutoff - the model does not know recent enough information.

**3. Which rate limit is the binding constraint during this spike?**

- A. RPM - 40 requests/min is close to the 50 RPM cap.
- B. TPM - 40 requests x 4,300 tokens each = 172,000 tokens, which exceeds the 50,000 TPM cap.
- C. Neither - 40 requests and 172,000 tokens are both within limits.
- D. RPDAY - daily limits are always the binding constraint for spikes.

**4. What is the approximate daily cost reduction from enabling prompt caching on the system prompt, assuming a 90% cache hit rate?**

- A. No reduction - prompt caching only works on the user message, not the system prompt.
- B. About $0.20/day - a small amount not worth the implementation effort.
- C. About $19.44/day - caching 2,700 of the 3,000 system prompt tokens at 10x cheaper rate saves significantly.
- D. About $2.16/day - a modest savings worth considering at higher scale.

**5. What happens on 2025-12-01 if you have not migrated?**

- A. The model returns lower-quality responses but continues working.
- B. API calls to that model version begin failing with an error, causing an outage.
- C. Anthropic automatically upgrades your calls to the latest Haiku version.
- D. The model enters read-only mode and stops accepting new conversations.

**6. Is this correct, and what is the typical ratio of output to input token pricing?**

- A. Correct - most providers price input and output tokens identically.
- B. Incorrect - output tokens typically cost 3-5x more than input tokens because generation is sequential while input processing is parallel.
- C. Incorrect - input tokens typically cost 3-5x more than output tokens because embedding large contexts is compute-intensive.
- D. Correct - the ratio is exactly 1:1 but varies by provider tier.

**7. Which model is the right choice, and why?**

- A. Model A - the larger context window means it can handle larger outputs.
- B. Model B - the 64k max output (roughly 48,000 words) is close enough for a 50,000-word document, while the context window is irrelevant for a 500-word input.
- C. Model A - a larger context window always correlates with a larger max output.
- D. Model B - a 200k context window is always more efficient than 1M for short inputs.

**8. Is the knowledge cutoff a hard blocker for this use case?**

- A. Yes, always - the model cannot answer any questions about events after 2022-01.
- B. No, never - knowledge cutoff only affects general knowledge, not API behavior.
- C. It depends - the model cannot answer from its training data alone, but if you provide relevant 2023-2024 content via RAG or in the context window, it can reason over that provided content.
- D. No - knowledge cutoff applies only to non-English languages.

_Answers are in `checks.json` in the lesson directory._